In [1]:
import spacy
import numpy as np
import pandas as pd
csv_df = pd.read_csv("./Laptop_Train_v2.csv", encoding="utf-8")

csv_df = csv_df[[
    "id",
    "Sentence",
    "Aspect Term",
    "polarity"
]]

csv_df.columns = ["id", "sentence", "aspect", "polarity"]

In [2]:
csv_df.head()

,id,sentence,aspect,polarity
0,2339,I charge it at night and skip taking the cord ...,cord,neutral
1,2339,I charge it at night and skip taking the cord ...,battery life,positive
2,1316,The tech guy then said the service center does...,service center,negative
3,1316,The tech guy then said the service center does...,"""sales"" team",negative
4,1316,The tech guy then said the service center does...,tech guy,neutral


In [4]:
# Remove all rows with polarity 'conflict' as they are not useful for training
csv_df = csv_df[csv_df['polarity'] != 'conflict']

In [5]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

csv_df["label"] = csv_df["polarity"].map(label2id)

In [6]:
print(csv_df["label"].value_counts())

label
2    987
0    866
1    460
Name: count, dtype: int64


# Create Dependency Graph

In [ ]:
nlp = spacy.load("en_core_web_sm")

def build_dependency_graph(sentence):

    doc = nlp(sentence)

    tokens = [token.text for token in doc]

    n = len(tokens)

    adj = np.eye(n)

    for token in doc:

        adj[token.i][token.head.i] = 1
        adj[token.head.i][token.i] = 1

    return tokens, adj

In [61]:
tokens, adj = build_dependency_graph(
    "The food was amazing but the service was terrible"
)

print(tokens)
print(adj.shape)

['The', 'food', 'was', 'amazing', 'but', 'the', 'service', 'was', 'terrible']
(9, 9)


# Convert to PyG graph format

In [62]:
from collections import Counter

counter = Counter()

for sent in csv_df["sentence"]:
    counter.update(sent.lower().split())

vocab = {
    word:i+2
    for i,(word,_) in enumerate(counter.items())
}

vocab["<pad>"] = 0
vocab["<unk>"] = 1

In [63]:
import torch
from torch_geometric.data import Data


def sentence_to_graph(sentence, aspect, label):

    tokens, adj = build_dependency_graph(sentence)

    node_features = []

    for token in tokens:
        node_features.append(
            vocab.get(token.lower(), 1)
        )

    x = torch.tensor(node_features)

    rows, cols = np.where(adj > 0)

    edge_index = torch.tensor(
        [rows, cols],
        dtype=torch.long
    )

    # aspect position
    aspect_words = aspect.lower().split()

    aspect_mask = [
        1 if token.lower() in aspect_words else 0
        for token in tokens
    ]

    aspect_mask = torch.tensor(aspect_mask)

    y = torch.tensor([label])

    return Data(
        x=x,
        edge_index=edge_index,
        aspect_mask=aspect_mask,
        y=y
    )

In [64]:
graphs = []

for _, row in csv_df.iterrows():

    g = sentence_to_graph(
        row["sentence"],
        row["aspect"],
        row["label"]
    )

    graphs.append(g)

# Create BERT Embeddings

In [65]:
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv

class ABSAGCN(nn.Module):

    def __init__(
        self,
        vocab_size,
        emb_dim=300,
        hidden_dim=128,
        num_classes=3
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            emb_dim
        )

        self.gcn1 = GCNConv(
            emb_dim,
            hidden_dim
        )

        self.gcn2 = GCNConv(
            hidden_dim,
            hidden_dim
        )

        self.fc = nn.Linear(
            hidden_dim,
            num_classes
        )

    def forward(
        self,
        x,
        edge_index,
        aspect_mask
    ):

        x = self.embedding(x)

        x = self.gcn1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.gcn2(
            x,
            edge_index
        )

        aspect_nodes = x[
            aspect_mask.bool()
        ]

        aspect_repr = aspect_nodes.mean(
            dim=0
        )

        logits = self.fc(
            aspect_repr
        )

        return logits

# Training 

In [66]:
model = ABSAGCN(
    vocab_size=len(vocab)
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

criterion = nn.CrossEntropyLoss()

In [67]:
for epoch in range(20):

    model.train()

    total_loss = 0

    for graph in graphs:

        optimizer.zero_grad()

        logits = model(
            graph.x,
            graph.edge_index,
            graph.aspect_mask
        )

        # logits: [3]
        logits = logits.unsqueeze(0)  # [1,3]

        # target: [1]
        target = graph.y.long()

        loss = criterion(
            logits,
            target
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1:02d} | Loss = {total_loss/len(graphs):.4f}"
    )

Epoch 01 | Loss = nan
Epoch 02 | Loss = nan
Epoch 03 | Loss = nan
Epoch 04 | Loss = nan
Epoch 05 | Loss = nan
Epoch 06 | Loss = nan
Epoch 07 | Loss = nan
Epoch 08 | Loss = nan
Epoch 09 | Loss = nan
Epoch 10 | Loss = nan
Epoch 11 | Loss = nan
Epoch 12 | Loss = nan
Epoch 13 | Loss = nan
Epoch 14 | Loss = nan
Epoch 15 | Loss = nan
Epoch 16 | Loss = nan
Epoch 17 | Loss = nan
Epoch 18 | Loss = nan
Epoch 19 | Loss = nan
Epoch 20 | Loss = nan


# Inference 

In [68]:
id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

model.eval()

def predict_sentiment(sentence, aspect):

    graph = sentence_to_graph(
        sentence=sentence,
        aspect=aspect,
        label=0  # dummy label
    )

    with torch.no_grad():

        logits = model(
            graph.x,
            graph.edge_index,
            graph.aspect_mask
        )

        probs = torch.softmax(logits, dim=-1)

        pred = torch.argmax(probs).item()

    return {
        "aspect": aspect,
        "sentiment": id2label[pred],
        "confidence": probs[pred].item(),
        "probabilities": {
            "negative": probs[0].item(),
            "neutral": probs[1].item(),
            "positive": probs[2].item()
        }
    }

In [69]:
result = predict_sentiment(
    sentence="The food was amazing but the service was terrible",
    aspect="food"
)

print(result)

{'aspect': 'food', 'sentiment': 'negative', 'confidence': nan, 'probabilities': {'negative': nan, 'neutral': nan, 'positive': nan}}


In [70]:
for i, g in enumerate(graphs):

    if torch.isnan(g.x.float()).any():
        print("NaN in x", i)

    if torch.isnan(g.edge_index.float()).any():
        print("NaN in edge_index", i)

    if g.aspect_mask.sum() == 0:
        print("NO ASPECT FOUND", i)

    if g.y.item() not in [0, 1, 2]:
        print("BAD LABEL", i, g.y)

NO ASPECT FOUND 303
NO ASPECT FOUND 459
NO ASPECT FOUND 530
NO ASPECT FOUND 531
NO ASPECT FOUND 638
NO ASPECT FOUND 731
NO ASPECT FOUND 778
NO ASPECT FOUND 1133
NO ASPECT FOUND 1134
NO ASPECT FOUND 1171
NO ASPECT FOUND 1181
NO ASPECT FOUND 1290
NO ASPECT FOUND 1319
NO ASPECT FOUND 1320
NO ASPECT FOUND 1659
NO ASPECT FOUND 1793
NO ASPECT FOUND 1902
NO ASPECT FOUND 2052
NO ASPECT FOUND 2053
NO ASPECT FOUND 2229


In [71]:
idx = 303

print(csv_df.iloc[idx]["sentence"])
print(csv_df.iloc[idx]["aspect"])

Yeah, of course smarty pants "fix it now")Software - Compared to the early 2011 edition I did see inbuilt applications crashing and it prompted me to send the report to Apple (which I promptly did).
Software


In [8]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple is looking at buying a UK startup.")

for ent in doc.ents:
    print(ent.text, ent.label_)

Apple ORG
UK GPE


In [9]:
for token in doc:
    print(token.text, token.dep_, token.head.text)

Apple nsubj looking
is aux looking
looking ROOT looking
at prep looking
buying pcomp at
a det startup
UK compound startup
startup dobj buying
. punct looking
